In [1]:
import os
import time
import numpy as np
import cv2
import tflite_runtime.interpreter as tflite
from reachy_sdk import ReachySDK

# 1. Configuration
ROBOT_IP = '10.22.129.133' 
MODEL_PATH = 'reachy_classifier.tflite'
LABELS_PATH = 'reachy_labels.txt'
SNAPSHOT_PATH = os.path.expanduser('~/dev/reachy-2026-iitg/reachy-tabletop-ai/live_view.jpg')

# 2. Read Labels
labels = {}
if os.path.exists(LABELS_PATH):
    with open(LABELS_PATH, 'r') as f:
        for line in f:
            pair = line.strip().split(maxsplit=1)
            if len(pair) == 2:
                labels[int(pair[0])] = pair[1]
else:
    labels = {0: "empty", 1: "cube", 2: "cylinder"}

# 3. Connect to Reachy
print(f"Connecting to Reachy at {ROBOT_IP}...")
try:
    robot = ReachySDK(host=ROBOT_IP)
    print("Successfully connected to the robot!")
except Exception as e:
    print(f"Connection failed: {e}")
    exit()

camera_feed = robot.right_camera 

# 4. Initialize TFLite Runtime
print("Loading model onto Edge TPU...")
try:
    interpreter = tflite.Interpreter(
        model_path=MODEL_PATH,
        experimental_delegates=[tflite.load_delegate('libedgetpu.so.1')]
    )
except Exception as e:
    print(f"Fallback to CPU mode: {e}")
    interpreter = tflite.Interpreter(model_path=MODEL_PATH)

interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

_, required_height, required_width, _ = input_details[0]['shape']

print("\n--- Live Classification Pipeline + Snapshot View Started ---")
print(f"Open this file on your PC to see the live feed: {SNAPSHOT_PATH}")
print("Press Ctrl+C to quit.\n")

try:
    while True:
        # Using your exact working camera loop sequence
        frame = camera_feed.last_frame
        if frame is None:
            continue
            
        # Create a deep copy for drawing so we don't ruin the source frame data
        display_frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR).copy()
        h_orig, w_orig, _ = display_frame.shape

        # 5. Prepare Image Frame for MobileNet (224x224)
        input_frame = cv2.resize(frame, (required_width, required_height))
        
        if input_details[0]['dtype'] == np.uint8:
            input_data = np.expand_dims(input_frame, axis=0).astype(np.uint8)
        else:
            normalized = (input_frame.astype(np.float32) - 128.0) / 128.0
            input_data = np.expand_dims(normalized, axis=0)

        # 6. Run Inference
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        
        output_data = interpreter.get_tensor(output_details[0]['index']).flatten()
        
        if output_data.dtype == np.uint8:
            scale, zero_point = output_details[0]['quantization']
            output_data = (output_data.astype(np.float32) - zero_point) * scale
            
        # Restrict evaluation strictly to your 3 custom training classes
        output_data = output_data[:3]
            
        exp_scores = np.exp(output_data - np.max(output_data))
        probabilities = exp_scores / exp_scores.sum()

        top_class_id = np.argmax(probabilities)
        confidence = probabilities[top_class_id]
        class_name = labels.get(top_class_id, f"unknown ({top_class_id})")

        print(f"Reachy Sees: {class_name.upper()} | Confidence: {confidence*100:.1f}%")

        # 7. Write the Snapshot out to your disk safely
        # FIX: Draw a small text bar strictly at the top 40 pixels of the image
        cv2.rectangle(display_frame, (0, 0), (w_orig, 40), (0, 0, 0), -1)
        text_display = f"AI Prediction: {class_name.upper()} ({confidence*100:.1f}%)"
        cv2.putText(display_frame, text_display, (15, 28), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
        
        # Save the crisp, unmasked frame over itself
        cv2.imwrite(SNAPSHOT_PATH, display_frame)
        
        time.sleep(0.5)

except KeyboardInterrupt:
    print("\nStopped by user.")


/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Connecting to Reachy at 10.22.129.133...
Successfully connected to the robot!
Loading model onto Edge TPU...

--- Live Classification Pipeline + Snapshot View Started ---
Open this file on your PC to see the live feed: /home/reachy/dev/reachy-2026-iitg/reachy-tabletop-ai/live_view.jpg
Press Ctrl+C to quit.

Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%
Reachy Sees: EMPTY | Confidence: 33.3%

Stopped by user.
